# Bonus experiments: feature selection under lesion-level evaluation on HAM10000

Companion notebook of the IEEE paper draft. It **does not replace** `Data_Science_annotated.ipynb`; it needs the files that notebook saved to Drive (`features_handcrafted.npz`, `features_phase1.npz`).

**What it adds (the new results the paper needs):**

| Experiment | Question | Output |
|---|---|---|
| E1 Repeated lesion-level splits | Are the differences between selection methods real, or noise from one split? | `res_multiseed.csv`, `tab_multiseed.tex`, `tab_wilcoxon.tex`, boxplot |
| E2 Metaheuristics with equal budget | GA vs binary PSO vs NSGA-II (multi-objective) | convergence plot, Pareto fronts, `tab_pareto.tex` |
| E3 Selection stability | Do the methods pick the same features on different splits? | stability table and heatmap |
| E4 Feature-group ablation | Which feature groups carry the signal? | `res_ablation.csv`, `tab_ablation.tex` |

**Run order:** top to bottom. E1 is the slow cell (about 5 to 8 minutes per seed on Colab CPU); results are saved after every seed, so after a disconnect just re-run the setup cells and the E1 cell, which resumes. Start with `N_SEEDS = 5`, then raise it to 10 and run again (only the new seeds are computed).

In [ ]:
# ---- Setup ----
import os, json, time, warnings, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

OUT = Path(os.environ.get("HAM_OUT", "/content/drive/MyDrive/ham10000_processed"))
FIGB = OUT / "figs_bonus"; FIGB.mkdir(exist_ok=True, parents=True)
TEX = OUT / "bonus_tex"; TEX.mkdir(exist_ok=True, parents=True)

# ---- Experiment configuration ----
N_SEEDS   = int(os.environ.get("N_SEEDS", 5))    # raise to 10 once the first 5 are done
K_SUBSET  = 20          # subset size for filter / wrapper / RF-importance selectors
POP, GENS = int(os.environ.get("POP", 20)), int(os.environ.get("GENS", 15))   # same evaluation budget for GA / BPSO / NSGA-II
ALPHA     = 0.002       # per-feature penalty in the single-objective fitness
FIT_SUB   = 4000        # training rows used inside the fitness function
RF_TREES  = int(os.environ.get("RF_TREES", 300))                # classifier used for the final scores
WRAP_SUB  = 1500        # rows used by the wrapper methods
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
print("seeds:", N_SEEDS, "| budget:", POP, "x", GENS)

In [ ]:
# ---- Load the saved feature table (47 handcrafted features) ----
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif, f_classif, SequentialFeatureSelector
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

h = np.load(OUT / "features_handcrafted.npz", allow_pickle=True)
X_all, FEAT = h["X"].astype(np.float32), list(h["names"])
p1 = np.load(OUT / "features_phase1.npz", allow_pickle=True)
y_all, groups_all = p1["y"], p1["groups"]
NF = len(FEAT)
print(X_all.shape, "features:", NF, "| lesions:", len(np.unique(groups_all)))

# feature groups used in the ablation (E4)
def grp(name):
    if name.startswith(("sex_",)): return "sex"
    if name.startswith("localization_"): return "location"
    if name == "age": return "age"
    if name.endswith("_center_minus_border") or name.endswith("_center") or name.endswith("_border"): return "centre/border"
    if name.split("_")[0] in ("H", "S", "V"): return "HSV"
    if name.split("_")[0] in ("R", "G", "B"): return "RGB stats"
    return "texture"
GROUP_OF = np.array([grp(n) for n in FEAT])
print(pd.Series(GROUP_OF).value_counts().to_dict())

In [ ]:
# ---- Helpers: lesion-level split, evaluation, selectors ----
def lesion_split(seed, test_size=0.20):
    """Stratified split at the lesion level: all images of a lesion go to the same side."""
    first = pd.DataFrame({"g": groups_all, "y": y_all}).drop_duplicates("g")
    tr_g, te_g = train_test_split(first["g"].values, test_size=test_size, stratify=first["y"].values, random_state=seed)
    is_tr = np.isin(groups_all, tr_g)
    assert set(groups_all[is_tr]).isdisjoint(set(groups_all[~is_tr]))
    return is_tr

def evaluate(Xtr, ytr, Xte, yte, cols, seed, trees=None):
    rf = RandomForestClassifier(trees or RF_TREES, min_samples_leaf=3, class_weight="balanced_subsample",
                                n_jobs=-1, random_state=seed).fit(Xtr[:, cols], ytr)
    p = rf.predict(Xte[:, cols]); pr = rf.predict_proba(Xte[:, cols])
    ok = len(rf.classes_) == 7 and len(np.unique(yte)) == 7
    auc = roc_auc_score(yte, pr, multi_class="ovr", average="macro", labels=rf.classes_) if ok else np.nan
    return dict(test_f1=f1_score(yte, p, average="macro"), test_acc=accuracy_score(yte, p), auc=auc,
                train_f1=f1_score(ytr, rf.predict(Xtr[:, cols]), average="macro"))

def make_fitness(Xtr, ytr, gtr, seed):
    """Inner lesion-grouped hold-out inside the TRAINING set: the test set is never touched by any selector."""
    tr_i, va_i = next(GroupShuffleSplit(1, test_size=0.25, random_state=seed).split(Xtr, ytr, gtr))
    if len(tr_i) > FIT_SUB: tr_i = np.random.default_rng(seed).choice(tr_i, FIT_SUB, replace=False)
    Xa, ya, Xb, yb = Xtr[tr_i], ytr[tr_i], Xtr[va_i], ytr[va_i]
    cache = {}
    def f1_of(mask):
        key = mask.tobytes()
        if key not in cache:
            if not mask.any(): cache[key] = 0.0
            else:
                rf = RandomForestClassifier(30, max_depth=8, max_samples=0.6, class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=seed).fit(Xa[:, mask], ya)
                cache[key] = f1_score(yb, rf.predict(Xb[:, mask]), average="macro")
        return cache[key]
    return f1_of, cache

def fit_penalised(f1_of):
    return lambda m: f1_of(m) - ALPHA * m.sum()

In [ ]:
# ---- Metaheuristics: GA, binary PSO, NSGA-II (all share the same fitness evaluations budget) ----
def run_ga(n, fit, pop=POP, gens=GENS, p_mut=0.03, seed=0):
    r = np.random.default_rng(seed); P = r.random((pop, n)) < 0.5; hist = []
    for g in range(gens):
        f = np.array([fit(x) for x in P]); hist.append(f.max())
        o = np.argsort(f)[::-1]; new = [P[o[0]].copy(), P[o[1]].copy()]            # elitism
        pick = lambda: P[(lambda c: c[np.argmax(f[c])])(r.choice(pop, 3, replace=False))]
        while len(new) < pop:
            c = np.where(r.random(n) < 0.5, pick(), pick()) ^ (r.random(n) < p_mut)
            if not c.any(): c[r.integers(n)] = True
            new.append(c)
        P = np.array(new)
    f = np.array([fit(x) for x in P]); return P[np.argmax(f)], hist

def run_bpso(n, fit, pop=POP, iters=GENS, seed=0, w_max=0.9, w_min=0.4, c1=2.0, c2=2.0, vmax=4.0):
    """Binary PSO (Kennedy & Eberhart, 1997): sigmoid transfer of the velocity gives the bit probability."""
    r = np.random.default_rng(seed); X = r.random((pop, n)) < 0.5; V = r.uniform(-1, 1, (pop, n))
    pf = np.array([fit(x) for x in X]); pb = X.copy(); gb = pb[pf.argmax()].copy(); gf = pf.max(); hist = [gf]
    for t in range(1, iters):
        w = w_max - (w_max - w_min) * t / max(1, iters - 1)
        Xf = X.astype(float)
        V = np.clip(w * V + c1 * r.random((pop, n)) * (pb.astype(float) - Xf) + c2 * r.random((pop, n)) * (gb.astype(float) - Xf), -vmax, vmax)
        X = r.random((pop, n)) < 1 / (1 + np.exp(-V))
        for i in range(pop):
            if not X[i].any(): X[i, r.integers(n)] = True
        f = np.array([fit(x) for x in X]); b = f > pf; pb[b], pf[b] = X[b], f[b]
        if pf.max() > gf: gf = pf.max(); gb = pb[pf.argmax()].copy()
        hist.append(gf)
    return gb, hist

def nd_sort(F):
    n = len(F); S = [[] for _ in range(n)]; cnt = np.zeros(n, int); fronts = [[]]
    for p in range(n):
        for q in range(n):
            if p == q: continue
            if np.all(F[p] <= F[q]) and np.any(F[p] < F[q]): S[p].append(q)
            elif np.all(F[q] <= F[p]) and np.any(F[q] < F[p]): cnt[p] += 1
        if cnt[p] == 0: fronts[0].append(p)
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                cnt[q] -= 1
                if cnt[q] == 0: nxt.append(q)
        i += 1; fronts.append(nxt)
    return fronts[:-1]

def crowd(F, idx):
    d = np.zeros(len(idx))
    for m in range(F.shape[1]):
        o = np.argsort(F[idx, m]); d[o[0]] = d[o[-1]] = np.inf
        rg = F[idx[o[-1]], m] - F[idx[o[0]], m]
        if rg == 0: continue
        for k in range(1, len(idx) - 1): d[o[k]] += (F[idx[o[k + 1]], m] - F[idx[o[k - 1]], m]) / rg
    return d

def run_nsga2(n, f1_of, pop=POP, gens=GENS, p_mut=0.03, seed=0):
    """NSGA-II (Deb et al., 2002). Objectives: maximise macro-F1, minimise the number of features."""
    r = np.random.default_rng(seed); P = r.random((pop, n)) < 0.5
    obj = lambda M: np.array([[-f1_of(m), m.sum()] for m in M])
    F = obj(P); hist = []
    for g in range(gens):
        fronts = nd_sort(F); rank = np.zeros(len(P), int)
        for k, fr in enumerate(fronts): rank[fr] = k
        cd = np.zeros(len(P))
        for fr in fronts: cd[fr] = crowd(F, np.array(fr))
        def tour():
            a, b = r.choice(len(P), 2, replace=False)
            return P[a] if (rank[a], -cd[a]) < (rank[b], -cd[b]) else P[b]
        Q = []
        while len(Q) < pop:
            c = np.where(r.random(n) < 0.5, tour(), tour()) ^ (r.random(n) < p_mut)
            if not c.any(): c[r.integers(n)] = True
            Q.append(c)
        Q = np.array(Q); R = np.vstack([P, Q]); FR = np.vstack([F, obj(Q)])
        nf = nd_sort(FR); sel = []
        for fr in nf:
            if len(sel) + len(fr) <= pop: sel += fr
            else:
                d = crowd(FR, np.array(fr)); sel += list(np.array(fr)[np.argsort(-d)[: pop - len(sel)]]); break
        P, F = R[sel], FR[sel]; hist.append(-F[:, 0].max())
    front = np.array(nd_sort(F)[0]); return P[front], -F[front, 0], F[front, 1].astype(int), hist

def knee(masks, f1s, sizes, tol=0.01):
    """Smallest subset whose inner-validation F1 is within `tol` of the best on the Pareto front."""
    ok = np.where(f1s >= f1s.max() - tol)[0]; return masks[ok[np.argmin(sizes[ok])]]

In [ ]:
# ---- Filter / wrapper / embedded selectors (all fitted on the TRAINING rows only) ----
def sel_filter_mi(Xtr, ytr, k, seed): return list(np.argsort(mutual_info_classif(Xtr, ytr, random_state=seed))[::-1][:k])
def sel_filter_f(Xtr, ytr, k):       return list(np.argsort(f_classif(Xtr, ytr)[0])[::-1][:k])

def sel_wrapper(Xtr, ytr, gtr, k, direction, seed):
    sub = np.random.default_rng(seed).choice(len(Xtr), min(WRAP_SUB, len(Xtr)), replace=False)
    cv = list(GroupKFold(3).split(Xtr[sub], ytr[sub], gtr[sub]))
    sfs = SequentialFeatureSelector(LogisticRegression(max_iter=300, class_weight="balanced"), n_features_to_select=k,
                                    direction=direction, scoring="f1_macro", cv=cv, n_jobs=-1).fit(Xtr[sub], ytr[sub])
    return list(np.where(sfs.get_support())[0])

def sel_embedded(Xtr, ytr, kind, seed, max_feats=25):
    best = None
    for C in [0.002, 0.005, 0.01, 0.02, 0.05, 0.1]:
        kw = dict(solver="saga", C=C, max_iter=200, tol=1e-3, class_weight="balanced", random_state=seed)
        m = LogisticRegression(penalty="l1", **kw) if kind == "L1" else LogisticRegression(penalty="elasticnet", l1_ratio=0.5, **kw)
        nz = np.where(np.abs(m.fit(Xtr, ytr).coef_).max(0) > 1e-6)[0]
        if best is None or 0 < len(nz) <= max_feats: best = list(nz)
    return best

def sel_rf_importance(Xtr, ytr, k, seed):
    rf = RandomForestClassifier(200, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=seed).fit(Xtr, ytr)
    return list(np.argsort(rf.feature_importances_)[::-1][:k])

## E1 + E2 + E3: repeated lesion-level splits with all selection methods

For each seed: a new stratified lesion-level 80/20 split, a scaler fitted on the training rows, all selectors run on the training rows only, and every selected subset scored once on the held-out test lesions with the same Random Forest. Per-seed results are appended to `res_multiseed.csv`, the selected feature masks to `selected_masks.json`, and the metaheuristic histories and Pareto fronts to `meta_hist.json`.

In [ ]:
RES_CSV, MASK_JSON, HIST_JSON = OUT / "res_multiseed.csv", OUT / "selected_masks.json", OUT / "meta_hist.json"
res_rows = pd.read_csv(RES_CSV).to_dict("records") if RES_CSV.exists() else []
masks = json.load(open(MASK_JSON)) if MASK_JSON.exists() else {}
hists = json.load(open(HIST_JSON)) if HIST_JSON.exists() else {}
done = {int(r["seed"]) for r in res_rows}

for seed in range(N_SEEDS):
    if seed in done: continue
    t0 = time.time()
    is_tr = lesion_split(seed)
    sc = StandardScaler().fit(X_all[is_tr]); Xs = sc.transform(X_all)
    Xtr, Xte, ytr, yte, gtr = Xs[is_tr], Xs[~is_tr], y_all[is_tr], y_all[~is_tr], groups_all[is_tr]
    f1_of, cache = make_fitness(Xtr, ytr, gtr, seed); fit = fit_penalised(f1_of)
    subsets, info = {}, {}
    subsets["All features"] = list(range(NF))
    subsets["Filter: MI"] = sel_filter_mi(Xtr, ytr, K_SUBSET, seed)
    subsets["Filter: ANOVA-F"] = sel_filter_f(Xtr, ytr, K_SUBSET)
    subsets["Wrapper: forward"] = sel_wrapper(Xtr, ytr, gtr, K_SUBSET, "forward", seed)
    subsets["Wrapper: backward"] = sel_wrapper(Xtr, ytr, gtr, K_SUBSET, "backward", seed)
    subsets["Embedded: L1"] = sel_embedded(Xtr, ytr, "L1", seed)
    subsets["Embedded: Elastic Net"] = sel_embedded(Xtr, ytr, "EN", seed)
    subsets["Embedded: RF importance"] = sel_rf_importance(Xtr, ytr, K_SUBSET, seed)
    g, hg = run_ga(NF, fit, seed=seed);                    subsets["GA"] = list(np.where(g)[0]);   info["GA"] = hg
    b, hb = run_bpso(NF, fit, seed=seed);                  subsets["BPSO"] = list(np.where(b)[0]); info["BPSO"] = hb
    M, F1s, sizes, hn = run_nsga2(NF, f1_of, seed=seed)
    subsets["NSGA-II (knee)"] = list(np.where(knee(M, F1s, sizes))[0]); info["NSGA-II"] = hn
    info["pareto"] = {"f1": [float(v) for v in F1s], "size": [int(v) for v in sizes]}
    for name, cols in subsets.items():
        r = evaluate(Xtr, ytr, Xte, yte, cols, seed); r.update(seed=seed, method=name, n_features=len(cols)); res_rows.append(r)
    masks[str(seed)] = {k: [int(i) for i in v] for k, v in subsets.items()}
    hists[str(seed)] = info
    pd.DataFrame(res_rows).to_csv(RES_CSV, index=False)
    json.dump(masks, open(MASK_JSON, "w")); json.dump(hists, open(HIST_JSON, "w"))
    print(f"seed {seed} done in {time.time()-t0:.0f}s | " + ", ".join(f"{k}: {v}" for k, v in
          pd.DataFrame(res_rows).query("seed == @seed").set_index("method")["test_f1"].round(3).items()))

### Summary table and statistical tests
Mean ± standard deviation over seeds, the Friedman test across all methods, and two-sided Wilcoxon signed-rank tests of every method against *All features*, with Holm correction. With few seeds the Wilcoxon test cannot reach p < 0.05 (n = 5 gives a minimum p of 0.0625), which is why the notebook aims for 10 seeds.

In [ ]:
from scipy import stats
R = pd.DataFrame(res_rows); seeds = sorted(R.seed.unique()); print("seeds available:", len(seeds))
order = ["All features", "Filter: MI", "Filter: ANOVA-F", "Wrapper: forward", "Wrapper: backward", "Embedded: L1",
         "Embedded: Elastic Net", "Embedded: RF importance", "GA", "BPSO", "NSGA-II (knee)"]
P = R.pivot(index="seed", columns="method", values="test_f1")[order]
S = R.groupby("method").agg(n=("n_features", "mean"), f1=("test_f1", "mean"), f1s=("test_f1", "std"), acc=("test_acc", "mean"),
                            auc=("auc", "mean"), gap=("train_f1", "mean")).loc[order]
S["gap"] = S["gap"] - S["f1"]
S["rank"] = P.rank(axis=1, ascending=False).mean()
print(S.round(3).to_string())
if len(seeds) >= 3:
    fr = stats.friedmanchisquare(*[P[m].values for m in order]); print("\nFriedman chi2 = %.2f, p = %.4g" % fr)
W = []
for m in order[1:]:
    d = P[m] - P["All features"]
    try: p = stats.wilcoxon(d, zero_method="wilcox").pvalue if (d != 0).any() else 1.0
    except ValueError: p = 1.0
    W.append((m, d.mean(), p))
W = pd.DataFrame(W, columns=["method", "mean_diff_vs_all", "p"]).sort_values("p")
W["p_holm"] = np.minimum(1, W["p"].values * (len(W) - np.arange(len(W)))); W["p_holm"] = np.maximum.accumulate(W["p_holm"].values)
print(); print(W.round(4).to_string(index=False))
S.round(4).to_csv(OUT / "summary_multiseed.csv"); W.round(5).to_csv(OUT / "wilcoxon_vs_all.csv", index=False)

In [ ]:
# ---- Figures: boxplot of test macro-F1, accuracy-vs-size trade-off, convergence, Pareto fronts ----
plt.rcParams.update({"font.size": 9})
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.boxplot([P[m].values for m in order], showmeans=True); ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels([m.replace(": ", ":\n") for m in order])
ax.set_ylabel("test macro-F1"); ax.set_title(f"Repeated lesion-level 80/20 splits (n = {len(seeds)} seeds)"); plt.xticks(fontsize=6.5)
plt.tight_layout(); plt.savefig(FIGB / "multiseed_box.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(3.4, 3.0))
ax.errorbar(S["n"], S["f1"], yerr=S["f1s"], fmt="o", capsize=2)
for m, r in S.iterrows(): ax.annotate(m.split(":")[-1].strip(), (r["n"], r["f1"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("mean # features"); ax.set_ylabel("test macro-F1"); plt.tight_layout(); plt.savefig(FIGB / "tradeoff.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(3.4, 2.8))
for name in ["GA", "BPSO", "NSGA-II"]:
    H = np.array([hists[str(s)][name] for s in seeds if str(s) in hists]); ax.plot(H.mean(0), label=name)
    ax.fill_between(range(H.shape[1]), H.mean(0) - H.std(0), H.mean(0) + H.std(0), alpha=0.15)
ax.set_xlabel("generation / iteration"); ax.set_ylabel("best inner-validation fitness"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIGB / "convergence_meta.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(3.4, 2.8))
for s in seeds:
    pf = hists[str(s)]["pareto"]; o = np.argsort(pf["size"]); ax.plot(np.array(pf["size"])[o], np.array(pf["f1"])[o], "o-", ms=3, alpha=0.6)
ax.set_xlabel("# selected features"); ax.set_ylabel("inner-validation macro-F1"); ax.set_title("NSGA-II Pareto fronts (one line per seed)", fontsize=8)
plt.tight_layout(); plt.savefig(FIGB / "pareto.png", dpi=300); plt.show()

In [ ]:
# ---- E3: selection stability (mean pairwise Jaccard across seeds) and selection-frequency heatmap ----
def jaccard(a, b): a, b = set(a), set(b); return len(a & b) / len(a | b)
stab = {}
for m in order[1:]:
    sets = [masks[str(s)][m] for s in seeds if str(s) in masks]
    pairs = [jaccard(a, b) for a, b in itertools.combinations(sets, 2)]; stab[m] = np.mean(pairs) if pairs else np.nan
stab = pd.Series(stab, name="stability"); print(stab.round(3).to_string())
stab.round(4).to_csv(OUT / "stability.csv")

freq = pd.DataFrame({m: np.bincount(np.concatenate([masks[str(s)][m] for s in seeds if str(s) in masks]), minlength=NF) / len(seeds)
                     for m in ["Filter: MI", "Wrapper: backward", "Embedded: L1", "Embedded: RF importance", "GA", "BPSO", "NSGA-II (knee)"]}, index=FEAT)
freq = freq.loc[freq.mean(1).sort_values(ascending=False).index[:25]]
fig, ax = plt.subplots(figsize=(3.6, 5.2)); im = ax.imshow(freq.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_yticks(range(len(freq))); ax.set_yticklabels(freq.index, fontsize=6); ax.set_xticks(range(freq.shape[1]))
ax.set_xticklabels([c.replace(": ", ":\n") for c in freq.columns], fontsize=5.5, rotation=90); plt.colorbar(im, ax=ax, label="selection frequency")
plt.tight_layout(); plt.savefig(FIGB / "stability_heatmap.png", dpi=300); plt.show()

## E4: feature-group ablation
Leave-one-group-out and single-group-only Random Forests over the same seeds (no selection involved, so this is fast).

In [ ]:
AB_CSV = OUT / "res_ablation.csv"
ab_rows = pd.read_csv(AB_CSV).to_dict("records") if AB_CSV.exists() else []
ab_done = {int(r["seed"]) for r in ab_rows}; groups_list = list(dict.fromkeys(GROUP_OF))
for seed in range(N_SEEDS):
    if seed in ab_done: continue
    is_tr = lesion_split(seed); Xs = StandardScaler().fit(X_all[is_tr]).transform(X_all)
    Xtr, Xte, ytr, yte = Xs[is_tr], Xs[~is_tr], y_all[is_tr], y_all[~is_tr]
    variants = {"All features": np.arange(NF)}
    for gname in groups_list:
        variants[f"without {gname}"] = np.where(GROUP_OF != gname)[0]; variants[f"only {gname}"] = np.where(GROUP_OF == gname)[0]
    for name, cols in variants.items():
        r = evaluate(Xtr, ytr, Xte, yte, list(cols), seed, trees=150); r.update(seed=seed, variant=name, n_features=len(cols)); ab_rows.append(r)
    pd.DataFrame(ab_rows).to_csv(AB_CSV, index=False); print("ablation seed", seed, "done")
A = pd.DataFrame(ab_rows).groupby("variant").agg(n=("n_features", "first"), f1=("test_f1", "mean"), f1s=("test_f1", "std"), acc=("test_acc", "mean"))
base = A.loc["All features", "f1"]; A["delta"] = A["f1"] - base
print(A.sort_values("f1", ascending=False).round(3).to_string())

## Export: LaTeX tables for the paper
Writes `tab_multiseed.tex`, `tab_wilcoxon.tex`, `tab_ablation.tex` and `bonus_macros.tex` into `ham10000_processed/bonus_tex/`. Upload these together with the figures from `figs_bonus/` into the Overleaf project next to `main.tex`; the paper picks them up automatically.

In [ ]:
def esc(s): return s.replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")
def pm(m, s): return f"{m:.3f}$\\pm${s:.3f}"

L = [r"\begin{table}[t]", r"\caption{Repeated lesion-level 80/20 splits (%d seeds): mean $\pm$ std of the test scores; the mean rank is over seeds (1 = best).}" % len(seeds),
     r"\label{tab:multiseed}", r"\centering\footnotesize", r"\begin{tabular}{@{}lccccc@{}}", r"\toprule",
     r"Method & \# feat. & Macro-F1 & Accuracy & AUC & Rank \\ \midrule"]
for m in order:
    r = S.loc[m]; L.append(f"{esc(m)} & {r['n']:.1f} & {pm(r['f1'], r['f1s'])} & {r['acc']:.3f} & {r['auc']:.3f} & {r['rank']:.1f} \\\\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(TEX / "tab_multiseed.tex", "w").write("\n".join(L))

L = [r"\begin{table}[t]", r"\caption{Wilcoxon signed-rank tests against \emph{All features} (Holm-adjusted p-values).}", r"\label{tab:wilcoxon}",
     r"\centering\footnotesize", r"\begin{tabular}{@{}lcc@{}}", r"\toprule", r"Method & Mean $\Delta$F1 & $p_{\mathrm{Holm}}$ \\ \midrule"]
for _, r in W.iterrows(): L.append(f"{esc(r['method'])} & {r['mean_diff_vs_all']:+.3f} & {r['p_holm']:.3f} \\\\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(TEX / "tab_wilcoxon.tex", "w").write("\n".join(L))

L = [r"\begin{table}[t]", r"\caption{Feature-group ablation (test macro-F1, mean $\pm$ std over seeds).}", r"\label{tab:ablation}",
     r"\centering\footnotesize", r"\begin{tabular}{@{}lccc@{}}", r"\toprule", r"Variant & \# feat. & Macro-F1 & $\Delta$ vs all \\ \midrule"]
for v, r in A.sort_values("f1", ascending=False).iterrows(): L.append(f"{esc(v)} & {int(r['n'])} & {pm(r['f1'], r['f1s'])} & {r['delta']:+.3f} \\\\")
L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(TEX / "tab_ablation.tex", "w").write("\n".join(L))

best = S["f1"].idxmax()
macros = {"NSeeds": len(seeds), "BestMethod": esc(best), "BestFscore": f"{S.loc[best,'f1']:.3f}", "AllFscore": f"{S.loc['All features','f1']:.3f}",
          "GAFscore": f"{S.loc['GA','f1']:.3f}", "BPSOFscore": f"{S.loc['BPSO','f1']:.3f}", "NSGAFscore": f"{S.loc['NSGA-II (knee)','f1']:.3f}",
          "GASize": f"{S.loc['GA','n']:.1f}", "NSGASize": f"{S.loc['NSGA-II (knee)','n']:.1f}"}
open(TEX / "bonus_macros.tex", "w").write("\n".join(f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in macros.items()))
print(open(TEX / "tab_multiseed.tex").read()); print(open(TEX / "bonus_macros.tex").read())